# Chương 7: Thực hành - Các nguyên lý cơ bản của Học máy (Machine Learning Basics)

**Mục tiêu bài thực hành:**
1. **Hiện tượng Khớp đa thức (Polynomial Fitting):** Trực quan hóa 3 trạng thái Thiếu khớp (Underfitting), Khớp tối ưu (Balanced) và Quá khớp (Overfitting).
2. **Mô phỏng Đánh đổi Thiên kiến - Phương sai (Bias-Variance Decomposition):** Huấn luyện 50 mô hình trên 50 tập dữ liệu độc lập để minh chứng bản chất toán học của Bias và Variance.
3. **Chế ngự Overfitting bằng Chính quy hóa (Regularization):** Tự cài đặt Ridge Regression ($L_2$) từ đầu và quan sát sự co cụm trọng số.
4. **Đo lường Hiệu năng trên Dữ liệu Mất cân bằng:** Tự cài đặt Ma trận nhầm lẫn (Confusion Matrix), Accuracy, Precision, Recall, $F_1$-Score từ đầu bằng `NumPy`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cố định hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Hiện tượng Khớp đa thức: Underfitting vs. Overfitting

Hàm thực tế tiềm ẩn: $y = \sin(2\pi x) + \epsilon$, với $\epsilon \sim \mathcal{N}(0, 0.04)$.  
Tập dữ liệu chỉ có $N = 15$ điểm mẫu. Ta mở rộng đặc trưng đa thức bậc $d$: $\mathbf{x} \mapsto [1, x, x^2, \dots, x^d]$ và tìm trọng số tối ưu bằng phương trình pháp tuyến: $\mathbf{w}^* = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$.

In [ ]:
def true_fn(x):
    return np.sin(2 * np.pi * x)

# Sinh 15 điểm huấn luyện
N = 15
X_train = np.sort(np.random.uniform(0, 1, N))
y_train = true_fn(X_train) + np.random.normal(0, 0.2, N)

# Hàm tạo ma trận đặc trưng đa thức bậc d
def get_poly_features(x, degree):
    return np.vstack([x**i for i in range(degree + 1)]).T

# Tập điểm mịn để vẽ đường cong dự đoán
x_dense = np.linspace(0, 1, 200)
degrees = [1, 3, 9]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
titles = [
    "Bậc 1: Thiếu khớp (Underfitting)\nHigh Bias, Dung lượng thấp",
    "Bậc 3: Khớp tối ưu (Balanced / Sweet Spot)\nKhớp tốt quy luật tiềm ẩn",
    "Bậc 9: Quá khớp (Overfitting)\nHigh Variance, Học cả nhiễu"
]

for idx, deg in enumerate(degrees):
    ax = axes[idx]
    Phi_train = get_poly_features(X_train, deg)
    # Nghiệm bình phương tối thiểu (thêm một lượng nhỏ 1e-10 vào đường chéo tránh suy biến)
    w = np.linalg.solve(Phi_train.T @ Phi_train + 1e-10 * np.eye(deg + 1), Phi_train.T @ y_train)
    
    Phi_dense = get_poly_features(x_dense, deg)
    y_dense_pred = Phi_dense @ w
    
    # Tính sai số huấn luyện MSE
    train_mse = np.mean((Phi_train @ w - y_train)**2)
    
    ax.plot(x_dense, true_fn(x_dense), 'g--', label='Hàm thực: sin(2pi*x)', linewidth=1.8)
    ax.plot(x_dense, y_dense_pred, 'r-', label=f'Mô hình đa thức (MSE={train_mse:.4f})', linewidth=2)
    ax.scatter(X_train, y_train, color='blue', edgecolors='k', s=45, label='Dữ liệu huấn luyện (N=15)')
    ax.set_title(titles[idx], fontsize=11, fontweight='bold')
    ax.set_ylim(-1.5, 1.5)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend(loc='lower left', fontsize=9)

plt.suptitle("Khảo sát Hiện tượng Underfitting và Overfitting qua Khớp Đa thức", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. Mô phỏng Đánh đổi Thiên kiến - Phương sai (Bias-Variance Simulation)

Ta sinh 40 tập dữ liệu huấn luyện độc lập từ cùng một phân phối thực tế.  
Với mỗi tập dữ liệu, ta huấn luyện hai mô hình: Mô hình bậc 1 và Mô hình bậc 9, rồi vẽ đồng thời 40 đường cong dự đoán để quan sát trực tiếp phương sai và thiên kiến.

In [ ]:
n_simulations = 40
preds_deg1 = []
preds_deg9 = []

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for _ in range(n_simulations):
    # Mỗi vòng sinh một tập dữ liệu ngẫu nhiên mới
    X_sim = np.random.uniform(0, 1, 15)
    y_sim = true_fn(X_sim) + np.random.normal(0, 0.2, 15)
    
    # Huấn luyện bậc 1
    Phi_1 = get_poly_features(X_sim, 1)
    w1 = np.linalg.solve(Phi_1.T @ Phi_1 + 1e-9 * np.eye(2), Phi_1.T @ y_sim)
    y_pred_1 = get_poly_features(x_dense, 1) @ w1
    preds_deg1.append(y_pred_1)
    ax1.plot(x_dense, y_pred_1, color='red', alpha=0.15)
    
    # Huấn luyện bậc 9
    Phi_9 = get_poly_features(X_sim, 9)
    w9 = np.linalg.solve(Phi_9.T @ Phi_9 + 1e-9 * np.eye(10), Phi_9.T @ y_sim)
    y_pred_9 = get_poly_features(x_dense, 9) @ w9
    preds_deg9.append(y_pred_9)
    ax2.plot(x_dense, y_pred_9, color='blue', alpha=0.15)

# Vẽ đường trung bình kỳ vọng E[f_hat]
mean_pred_1 = np.mean(preds_deg1, axis=0)
mean_pred_9 = np.mean(preds_deg9, axis=0)

ax1.plot(x_dense, true_fn(x_dense), 'g--', linewidth=2.5, label='Hàm thực tế f(x)')
ax1.plot(x_dense, mean_pred_1, 'k-', linewidth=2.5, label='Kỳ vọng dự đoán E[f_hat]')
ax1.set_title("Mô hình Bậc 1: High Bias, Low Variance\n(Các đường cong rất chụm nhưng xa hàm thực)", fontsize=11, fontweight='bold')
ax1.set_ylim(-1.5, 1.5)
ax1.legend()

ax2.plot(x_dense, true_fn(x_dense), 'g--', linewidth=2.5, label='Hàm thực tế f(x)')
ax2.plot(x_dense, mean_pred_9, 'k-', linewidth=2.5, label='Kỳ vọng dự đoán E[f_hat]')
ax2.set_title("Mô hình Bậc 9: Low Bias, High Variance\n(Đường trung bình rất sát hàm thực, nhưng từng mô hình dao động điên cuồng)", fontsize=11, fontweight='bold')
ax2.set_ylim(-2.0, 2.0)
ax2.legend()

plt.suptitle("Minh họa Trực quan Phân rã Thiên kiến - Phương sai qua 40 Lần thử nghiệm", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Khắc phục Overfitting bằng Chính quy hóa Ridge ($L_2$ Regularization)

Áp dụng số hạng phạt $\frac{\lambda}{2} \|\mathbf{w}\|_2^2$ lên bài toán đa thức bậc 9:
$$\mathbf{w}_{\text{ridge}}^* = (\mathbf{X}^T\mathbf{X} + \lambda \mathbf{I})^{-1} \mathbf{X}^T\mathbf{y}$$

In [ ]:
lambdas = [0.0, 1e-5, 1e-2, 1.0]
deg = 9
Phi_train = get_poly_features(X_train, deg)
Phi_dense = get_poly_features(x_dense, deg)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, lam in enumerate(lambdas):
    ax = axes[idx]
    # Không phạt trọng số bias w0
    reg_matrix = lam * np.eye(deg + 1)
    reg_matrix[0, 0] = 0.0
    
    w_reg = np.linalg.solve(Phi_train.T @ Phi_train + reg_matrix, Phi_train.T @ y_train)
    y_dense_pred = Phi_dense @ w_reg
    
    weight_norm = np.linalg.norm(w_reg)
    
    ax.plot(x_dense, true_fn(x_dense), 'g--', label='Hàm thực', linewidth=1.5)
    ax.plot(x_dense, y_dense_pred, 'r-', label='Mô hình', linewidth=2)
    ax.scatter(X_train, y_train, color='blue', edgecolors='k', s=35)
    ax.set_title(f"lambda = {lam}\nChuẩn ||w|| = {weight_norm:.2f}", fontsize=11, fontweight='bold')
    ax.set_ylim(-1.5, 1.5)
    if idx == 0:
        ax.legend(loc='lower left', fontsize=8)

plt.suptitle("Tác dụng của Chính quy hóa L2 (Ridge) trong việc Chế ngự Dao động của Mô hình Bậc 9", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Đo lường Hiệu năng trên Dữ liệu Mất cân bằng (Evaluation Metrics from Scratch)

Tạo tập dữ liệu y tế gồm 1000 bệnh nhân, trong đó chỉ có 30 người thực sự nhiễm bệnh ($3\%$ Dương tính).

In [ ]:
# Tạo nhãn thực tế: 970 ca âm tính (0), 30 ca dương tính (1)
y_true = np.array([0] * 970 + [1] * 30)

# 1. Mô hình ngây thơ (Naive Baseline): Luôn dự đoán 0 (Âm tính)
y_pred_naive = np.zeros_like(y_true)

# 2. Mô hình chuyên môn AI (Chẩn đoán được 24 ca bệnh, báo nhầm 20 ca)
y_pred_ai = y_true.copy()
# Mô hình AI bỏ sót 6 ca bệnh
y_pred_ai[np.where(y_true == 1)[0][:6]] = 0
# Mô hình AI báo nhầm 20 ca âm tính thành dương tính
y_pred_ai[np.where(y_true == 0)[0][:20]] = 1

def calculate_metrics(y_real, y_hat):
    tp = np.sum((y_real == 1) & (y_hat == 1))
    tn = np.sum((y_real == 0) & (y_hat == 0))
    fp = np.sum((y_real == 0) & (y_hat == 1))
    fn = np.sum((y_real == 1) & (y_hat == 0))
    
    acc = (tp + tn) / len(y_real)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
    
    return tp, fp, tn, fn, acc, prec, rec, f1

tp_n, fp_n, tn_n, fn_n, acc_n, prec_n, rec_n, f1_n = calculate_metrics(y_true, y_pred_naive)
tp_a, fp_a, tn_a, fn_a, acc_a, prec_a, rec_a, f1_a = calculate_metrics(y_true, y_pred_ai)

print("=== BẢNG ĐỐI CHIẾU ĐỘ ĐO TRÊN DỮ LIỆU MẤT CÂN BẰNG (3% BỆNH) ===")
print(f"Mô hình Ngây thơ (Đoán toàn 0) : Accuracy={acc_n*100:.1f}% | Precision={prec_n*100:.1f}% | Recall={rec_n*100:.1f}% | F1={f1_n:.4f}")
print(f"Mô hình Chuyên môn AI          : Accuracy={acc_a*100:.1f}% | Precision={prec_a*100:.1f}% | Recall={rec_a*100:.1f}% | F1={f1_a:.4f}")

# Nhận xét: Dù Accuracy của mô hình ngây thơ cao hơn (97% vs 97.4%), F1-Score và Recall của AI vượt trội hoàn toàn!

## 5. Tổng kết bài học thực hành

1. **Dung lượng mô hình:** Mô hình quá đơn giản $\implies$ High Bias (Underfitting); Mô hình quá phức tạp $\implies$ High Variance (Overfitting).
2. **Chính quy hóa $L_2$ (Ridge):** Bằng cách ghìm giữ độ lớn của các trọng số $\|\mathbf{w}\|_2$, ta hạn chế được tính dao động cực đoan của mô hình phức tạp mà vẫn giữ lại được khả năng uốn lượn mềm mại.
3. **Hiệu năng trên dữ liệu lệch:** Accuracy là một cái bẫy chết người trên dữ liệu mất cân bằng; các chỉ số Precision, Recall và $F_1$-Score mới là kim chỉ nam phản ánh độ tin cậy nghiệp vụ thực sự.